# CNU RAG 파이프라인 콜랩 실행 노트북

**순서**: pip → Qwen 로딩 → 컨텍스트 prefix 생성 → 벡터DB 재빌드 → 답변 생성(5 ablation) → Gemini judge → 비교표.

총 소요 (T4 기준): 컨텍스트 생성 1.5~3h + 벡터DB 30m + 답변 200×5조합 60m + judge 20m = **3~5시간**.

각 셀은 idempotent. checkpoint resume 지원. 노트북 idle timeout(90분) 피하려면 주기적 활동 필요.

In [ ]:
# 셀 1) Drive 마운트(선택) + git pull + pip install

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = '/content/cnu-llm-bot'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Longarden/cnu-llm-bot.git', REPO])
%cd $REPO
!git pull

# 최소 의존성. autoawq는 GPU 빌드 길어 느릴 수 있음.
!pip install -q transformers>=4.40.0 sentence-transformers chromadb rank_bm25 kiwipiepy trafilatura olefile pdfplumber PyPDF2 beautifulsoup4 requests autoawq accelerate google-genai

In [ ]:
# 셀 2) 환경변수 설정 (콜랩 secret 또는 직접 입력)
import os
from google.colab import userdata
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    os.environ['GEMINI_API_KEY'] = ''  # 직접 입력: os.environ['GEMINI_API_KEY'] = 'AIza...'
os.environ['RERANK'] = '1'
os.environ['JUDGE_MOCK'] = '0' if os.environ['GEMINI_API_KEY'] else '1'
print('GEMINI_API_KEY set:', bool(os.environ['GEMINI_API_KEY']))

In [ ]:
# 셀 3) Qwen 3B AWQ 로딩 + llm_call wrapper (컨텍스트 prefix·MQ·HyDE·self_verify에서 사용)
from transformers import AutoTokenizer
from awq import AutoAWQForCausalLM
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct-AWQ'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
qwen = AutoAWQForCausalLM.from_quantized(MODEL_ID, fuse_layers=True, device_map='cuda')

def llm_call(prompt: str, max_new_tokens: int = 200, temperature: float = 0.0) -> str:
    msgs = [{'role':'user','content':prompt}]
    inputs = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = qwen.generate(inputs, max_new_tokens=max_new_tokens,
                        do_sample=temperature>0, temperature=max(temperature, 1e-3),
                        pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print(llm_call('한 줄로 답해라: 1+1은?', max_new_tokens=20))

In [ ]:
# 셀 4) 데이터 로드 + 청킹 (400/50)
from embedding.data_loader import load_scoped_docs
from embedding.chunker import chunk_documents
docs = load_scoped_docs()
chunks = chunk_documents(docs)
print(f'docs {len(docs)}  chunks {len(chunks)}')

In [ ]:
# 셀 5) W1 Contextual Retrieval — 각 청크에 50~80자 컨텍스트 prefix 추가
# checkpoint_every=200, resume=True. 끊겨도 다시 실행하면 이어함. 1.5~3시간.
from retrieval.contextual_chunker import contextualize_chunks
CTX_PATH = 'data/contextual_chunks.json'
ctx_chunks = contextualize_chunks(docs, chunks, llm_call,
                                  out_path=CTX_PATH,
                                  checkpoint_every=200,
                                  resume=True)
ok = sum(1 for c in ctx_chunks if c.get('context_prefix'))
print(f'컨텍스트 prefix 생성: {ok}/{len(ctx_chunks)}')

In [ ]:
# 셀 6) 벡터DB 재빌드 (bge-m3 GPU). chroma_db는 콜랩 디스크 — Drive에 두려면 CHROMA_PATH 변경.
import chromadb
from embedding.vector_store import build_vector_db, count, CHROMA_PATH
client = chromadb.PersistentClient(path=CHROMA_PATH)
try:
    client.delete_collection('cnu_rag')
    print('기존 컬렉션 삭제')
except Exception as e:
    print(f'삭제 스킵: {e}')
build_vector_db(ctx_chunks)
print(f'벡터DB 청크: {count()}')

In [ ]:
# 셀 7) BM25 인덱스 + smoke test
from retrieval.hybrid_retriever import init_bm25_from_db, retrieve
init_bm25_from_db()
for q in ['오늘 1학생회관 점심 메뉴', '수강신청 언제부터야', '셔틀버스 시간표']:
    docs_r = retrieve(q, n_results=3, use_meta_boost=True)
    print(q, '→', [d.get('data_category') for d in docs_r])

In [ ]:
# 셀 8) RAG 답변 생성 wrapper (Qwen로 답)
from generation.prompt import SYSTEM_PROMPT, build_user_prompt

def llm_generate(question: str, retrieved_docs: list[dict]) -> str:
    user = build_user_prompt(question, retrieved_docs)
    msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':user}]
    inputs = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = qwen.generate(inputs, max_new_tokens=400, do_sample=False,
                        pad_token_id=tok.pad_token_id or tok.eos_token_id)
    raw = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    from interface.answer_questions import _clean_answer
    return _clean_answer(raw)

In [ ]:
# 셀 9) Ablation 실행 — 5조합 × 200문항 답변 생성
import json
from scripts.ablation_runner import run_ablation
questions = [json.loads(l) for l in open('data/testset.jsonl', encoding='utf-8') if l.strip()]
print(f'평가 질문 {len(questions)}건')
results = run_ablation(questions, llm_call=llm_call, llm_generate=llm_generate,
                       out_path='data/ablation_answers.json', n_results=5, verbose_every=25)
print({k: len(v) for k,v in results.items()})

In [ ]:
# 셀 10) Gemini judge 일괄 + 비교표
from eval.llm_judge import judge
from scripts.ablation_runner import score_ablation, print_scoreboard
scores = score_ablation(results, judge)
json.dump(scores, open('data/ablation_scores.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print_scoreboard(scores)

## 결과 해석 가이드

- `baseline` 대비 각 옵션이 어떤 축(accuracy/relevance/domain_compliance)에서 오르는지 확인.
- `+meta` 가 의미 있으면 학식·셔틀 같은 카테고리 강한 질의에서 효과.
- `+query` 효과는 표현이 다양한 질문(시간성/생활)에서 큼.
- `+crag_esc` 는 baseline에서 reject 비율이 높았던 항목 회복.
- `full` 이 다른 모든 조합보다 일관되게 높으면 채택.
- 어느 옵션이 다른 옵션을 깎는다면 (예: meta + query 동시 켜면 accuracy ↓) 해당 조합 제외.

최종 채택 조합으로 `interface/answer_questions.py` 의 retrieve 호출 옵션 고정 → 교수님 제출 노트북에 반영.